In [ ]:
import requests
import time
import random
import string
import tiktoken

# ==========================================
# [설정 영역]
# ==========================================
MODEL_NAME = "qwen3:30b-a3b" 
API_URL = "http://localhost:11434/api/generate"

# 테스트 반복 횟수 (신뢰도 향상을 위해 3회 반복 후 평균 계산)
ITERATIONS = 3  

TEST_CASES = [
    {"label": "hi (2)",    "target_tokens": 2,    "output_len": 10},
    {"label": "1K (969)",  "target_tokens": 969,  "output_len": 300},
    {"label": "2K (1930)", "target_tokens": 1930, "output_len": 300},
    {"label": "4K (3846)", "target_tokens": 3846, "output_len": 300},
    {"label": "8K (7678)", "target_tokens": 7678, "output_len": 300},
]

MAX_CONTEXT = 16384 
tokenizer = tiktoken.get_encoding("cl100k_base") 

# ==========================================
# [함수 영역]
# ==========================================
def generate_exact_prompt(target_tokens):
    """
    tiktoken을 사용하여 정확히 target_tokens 길이에 맞게 프롬프트를 자릅니다.
    """
    if target_tokens < 10:
        return "hi " * (target_tokens // 2 + 1)
        
    # 1. 캐시 무력화를 위한 무작위 난수(Nonce) 생성
    nonce = "".join(random.choices(string.ascii_letters + string.digits, k=16))
    prefix = f"System Nonce: {nonce}\n\n"
    prefix_tokens = tokenizer.encode(prefix)
    
    if target_tokens <= len(prefix_tokens):
        return prefix
        
    remaining_target = target_tokens - len(prefix_tokens)
    
    # 2. 반복할 베이스 텍스트
    base_text = "The deepseek coder is a powerful model for coding tasks and general reasoning benchmarks. It efficiently handles context and generation tasks with mixture of experts architecture. "
    base_tokens = tokenizer.encode(base_text)
    
    # 3. 필요한 토큰 수만큼 정확하게 리스트 슬라이싱
    repeat_count = (remaining_target // len(base_tokens)) + 1
    extended_tokens = (base_tokens * repeat_count)[:remaining_target]
    
    # 4. 토큰을 다시 문자열로 디코딩
    final_prompt = tokenizer.decode(prefix_tokens + extended_tokens)
    return final_prompt

def run_single_test(case):
    prompt = generate_exact_prompt(case["target_tokens"])
    
    payload = {
        "model": MODEL_NAME,
        "prompt": prompt,
        "stream": False,
        "options": {
            "num_predict": case["output_len"], 
            "num_ctx": MAX_CONTEXT,            
            "temperature": 0.0,                
            "num_gpu": 99                      
        }
    }

    try:
        response = requests.post(API_URL, json=payload, timeout=300).json()

        if "error" in response:
            return None, None, f"Error: {response['error']}"

        # Ollama 내부 토크나이저가 평가한 실제 토큰 수
        p_count = response.get('prompt_eval_count', 0)
        p_dur = response.get('prompt_eval_duration', 0)
        e_count = response.get('eval_count', 0)
        e_dur = response.get('eval_duration', 0)

        # Ollama에서 반환한 "실제 토큰 수"를 기준으로 Prefill 속도 계산 (가장 정확)
        prefill = (p_count / (p_dur / 1e9)) if p_dur > 0 else 0
        decode = (e_count / (e_dur / 1e9)) if e_dur > 0 else 0

        return prefill, decode, p_count

    except requests.exceptions.ConnectionError:
        return None, None, "CUDA OOM or Server Down"
    except Exception as e:
        return None, None, str(e)

# ==========================================
# [실행 영역]
# ==========================================
print(f"\n🚀 Starting Exact Benchmark for: {MODEL_NAME}")
print(f"📦 Max Context Setting: {MAX_CONTEXT} | 🔄 Iterations per case: {ITERATIONS}")
print("-" * 85)
print(f"{'Prompt Size':<15} | {'Avg Prefill (t/s)':<18} | {'Avg Decode (t/s)':<18} | {'Avg Actual Tokens'}")
print("-" * 85)

# 1. Warm-up
print(f"{'Warm-up...':<15} | {'...':<18} | {'...':<18} | Loading Model...")
requests.post(API_URL, json={"model": MODEL_NAME, "prompt": "hi", "stream": False})

# 2. 메인 벤치마크 루프
for case in TEST_CASES:
    prefill_speeds = []
    decode_speeds = []
    actual_tokens_list = []
    error_note = ""

    for i in range(ITERATIONS):
        p_speed, d_speed, act_tokens = run_single_test(case)
        
        if p_speed is None:  
            error_note = act_tokens 
            break
            
        prefill_speeds.append(p_speed)
        decode_speeds.append(d_speed)
        actual_tokens_list.append(act_tokens)
        
        time.sleep(0.5)

    if error_note:
        print(f"{case['label']:<15} | {'Fail':<18} | {'Fail':<18} | {error_note}")
    else:
        avg_prefill = sum(prefill_speeds) / len(prefill_speeds)
        avg_decode = sum(decode_speeds) / len(decode_speeds)
        avg_tokens = int(sum(actual_tokens_list) / len(actual_tokens_list))
        
        print(f"{case['label']:<15} | {avg_prefill:<18.2f} | {avg_decode:<18.2f} | {avg_tokens} tokens")

print("-" * 85)
print("✅ Benchmark Completed.")


🚀 Starting Exact Benchmark for: qwen3:30b-a3b
📦 Max Context Setting: 16384 | 🔄 Iterations per case: 3
-------------------------------------------------------------------------------------
Prompt Size     | Avg Prefill (t/s)  | Avg Decode (t/s)   | Avg Actual Tokens
-------------------------------------------------------------------------------------
Warm-up...      | ...                | ...                | Loading Model...
hi (2)          | 131.40             | 17.21              | 13 tokens
1K (969)        | 235.99             | 13.54              | 948 tokens
2K (1930)       | 224.94             | 12.31              | 1877 tokens
